# pivotai — Distilled Model (`pivotai-distill`)
**Session 2 of 3** — Trains on Phase 2 DeepSeek reasoning traces (450 records, 5 epochs).

**Platform**: Lightning.ai A100 (40GB VRAM, 200GB disk) — required for full reasoning chains (seq_len=16384).

**Actual results**: train_loss=0.429, final step loss=0.254 (5 epochs, 285 steps, 476s)

**Before running:**
1. Upload `pivotai.zip` (full project folder) to Lightning.ai studio storage
2. Paste HF write token directly (no Colab Secrets on Lightning.ai)
3. Run all cells in order — expected time ~8 min on A100

In [ ]:
# Cell 0 — Set memory env var FIRST (must run before any torch import)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('✓ CUDA alloc config set')

In [ ]:
# Cell 1 — Setup Lightning.ai paths + unzip project
import zipfile, os, sys

BASE = '/teamspace/studios/this_studio'
if not os.path.exists(f'{BASE}/travel_project'):
    with zipfile.ZipFile(f'{BASE}/pivotai.zip', 'r') as z:
        z.extractall(BASE)
    print('✓ Unzipped')
else:
    print('✓ Already extracted')

os.makedirs(f'{BASE}/pivotai_models', exist_ok=True)
print(f'Working directory: {BASE}')

In [ ]:
# Cell 2 — Add project to path + verify training files
import sys
sys.path.insert(0, f'{BASE}/travel_project')

TRAIN_FILE = f'{BASE}/travel_project/data/training/distill_train.jsonl'
VAL_FILE   = f'{BASE}/travel_project/data/training/distill_val.jsonl'

for f in [TRAIN_FILE, VAL_FILE]:
    lines = open(f).read().splitlines()
    print(f'{f.split("/")[-1]}: {len(lines)} records')

In [ ]:
# Cell 3 — Install dependencies (~5 min)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl peft accelerate bitsandbytes -q
print('✓ Install complete')

In [ ]:
# Cell 4 — Load Llama 3.1 8B with 4-bit QLoRA
# A100 (bfloat16 native) + seq_len=16384 to fit full 5k-token reasoning chains
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 16384

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-bnb-4bit',
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('✓ Model loaded with MAX_SEQ_LEN=16384 (A100 bf16)')

In [ ]:
# Cell 5 — Load dataset + filter sequences > MAX_SEQ_LEN tokens
# At 16384: keeps ~78% of records fully intact (median distill trace ~5226 tokens)
from datasets import load_dataset

ALPACA_PROMPT = """### Instruction:\n{}\n\n### Input:\n{}\n\n### Response:\n{}"""
EOS = tokenizer.eos_token

def format_alpaca(examples):
    return {'text': [
        ALPACA_PROMPT.format(i, inp, out) + EOS
        for i, inp, out in zip(examples['instruction'], examples['input'], examples['output'])
    ]}

def filter_length(examples):
    lengths = tokenizer(examples['text'], truncation=False, return_length=True, padding=False)['length']
    return [l <= MAX_SEQ_LEN for l in lengths]

train_ds = load_dataset('json', data_files=TRAIN_FILE, split='train').map(format_alpaca, batched=True)
val_ds   = load_dataset('json', data_files=VAL_FILE,   split='train').map(format_alpaca, batched=True)

before = len(train_ds)
train_ds = train_ds.filter(filter_length, batched=True)
print(f'✓ Train: {before} → {len(train_ds)} records fit within {MAX_SEQ_LEN} tokens')
print(f'  (dropped {before - len(train_ds)} records exceeding {MAX_SEQ_LEN} tokens)')
print(f'✓ Val: {len(val_ds)}')

In [ ]:
# Cell 6 — Train (5 epochs — compensates for smaller 450-record dataset)
# A100 requires bf16=True (not fp16) — bfloat16 is A100's native format
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=5,
        learning_rate=2e-4,
        warmup_steps=10,
        fp16=False,
        bf16=True,
        logging_steps=20,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir=f'{BASE}/pivotai_distill_checkpoints',
        save_strategy='no',
        report_to='none',
        dataloader_num_workers=0,
    ),
)

stats = trainer.train()
print(f'✓ Training complete — {stats.metrics["train_runtime"]:.0f}s ({stats.metrics["train_runtime"]/3600:.1f}h)')
print(f'  Final train_loss: {stats.metrics["train_loss"]:.4f}')

In [ ]:
# Cell 7 — Save LoRA locally, convert to GGUF, upload to HuggingFace
import shutil, json as _json
from huggingface_hub import HfApi

HF_TOKEN    = ''  # paste your HF write token here
HF_USERNAME = 'ishreyadev'

# Step 1: Save LoRA adapter
model.save_pretrained(f'{BASE}/pivotai_distill_lora')
tokenizer.save_pretrained(f'{BASE}/pivotai_distill_lora')
print('✓ LoRA saved')

# Step 2: Convert to GGUF locally (saves to pivotai_distill_gguf/)
model.save_pretrained_gguf(f'{BASE}/pivotai_distill', tokenizer, quantization_method='q4_k_m')
gguf_src = f'{BASE}/pivotai_distill_gguf/Meta-Llama-3.1-8B.Q4_K_M.gguf'
shutil.copy(gguf_src, f'{BASE}/pivotai_models/pivotai_distill.gguf')
print('✓ GGUF created and copied to pivotai_models/')

# Step 3: Upload LoRA to HF
model.push_to_hub(f'{HF_USERNAME}/pivotai-distill-lora', token=HF_TOKEN, private=False)
tokenizer.push_to_hub(f'{HF_USERNAME}/pivotai-distill-lora', token=HF_TOKEN)
print(f'✓ LoRA → huggingface.co/{HF_USERNAME}/pivotai-distill-lora')

# Step 4: Upload GGUF to HF
api = HfApi()
api.create_repo(f'{HF_USERNAME}/pivotai-distill-gguf', token=HF_TOKEN, exist_ok=True)
api.upload_file(
    path_or_fileobj=gguf_src,
    path_in_repo='pivotai_distill.Q4_K_M.gguf',
    repo_id=f'{HF_USERNAME}/pivotai-distill-gguf',
    token=HF_TOKEN,
)
print(f'✓ GGUF → huggingface.co/{HF_USERNAME}/pivotai-distill-gguf')
print('\npivotai-distill COMPLETE')